In [2]:
import cProfile
import pstats
import tracemalloc
from glom import glom, Iter, Coalesce, OMIT

# 1. Custom Profiler Class implementation
class Profiler:
    def __init__(self):
        self._profiler = cProfile.Profile()
        self._peak_memory = 0

    def start(self):
        # Reset and start memory tracking
        tracemalloc.stop()
        tracemalloc.start()
        # Start CPU profiling
        self._profiler.enable()

    def stop(self):
        # Stop CPU profiling
        self._profiler.disable()
        # Capture memory stats before stopping tracking
        _, peak = tracemalloc.get_traced_memory()
        self._peak_memory = peak / 1024  # Convert to KB
        tracemalloc.stop()

    def print(self, label="Execution Profile"):
        print(f"\n================ {label.upper()} ================")
        print(f"PEAK MEMORY USAGE: {self._peak_memory:.2f} KB")
        print("---------------- CPU TIME BREAKDOWN ----------------")
        
        # Sort and display the top execution functions by internal time
        stats = pstats.Stats(self._profiler)
        stats.strip_dirs().sort_stats(pstats.SortKey.TIME).print_stats(10)


# 2. Your Custom Recursive Function
def extract_values_from_list(data, path):
    keys = path.split(".")

    def _extract(current, keys):
        if not keys:
            return [current], True

        key = keys[0]
        rest = keys[1:]

        results = []
        found = False

        if isinstance(current, list):
            for item in current:
                vals, item_found = _extract(item, keys)
                if item_found:
                    found = True
                results.extend(vals)

        elif isinstance(current, dict):
            if key in current:
                found = True
                vals, child_found = _extract(current[key], rest)
                found = found or child_found
                results.extend(vals)

        return results, found

    return _extract(data, keys)


# 3. Setup Dataset and Test Variables (Looping execution to accumulate metrics)
mock_api_response = {
    "users": [
        {
            "profile": {
                "addresses": [{"city": f"City_{i}"}, {"city": f"Town_{i}"}]
            }
        } if i % 10 != 0 else {"profile": {}} 
        for i in range(2000)
    ]
}

path_string = "users.profile.addresses.city"

glom_spec = (
    'users', 
    [Coalesce('profile.addresses', default=OMIT)], 
    Iter().flatten(), 
    ['city']
)

# Scaling loop count to let the profiler collect stable data
RUN_LOOPS = 200


# 4. Profile the Custom Recursive Function
def run_custom_test():
    for _ in range(RUN_LOOPS):
        extract_values_from_list(mock_api_response, path_string)

profiler = Profiler()
profiler.start()

run_custom_test()

profiler.stop()
profiler.print(label="Custom Recursive Function Profile")


# 5. Profile the Glom Implementation
def run_glom_test():
    for _ in range(RUN_LOOPS):
        glom(mock_api_response, glom_spec)

profiler = Profiler()
profiler.start()

run_glom_test()

profiler.stop()
profiler.print(label="Glom Implementation Profile")



================ CUSTOM RECURSIVE FUNCTION PROFILE ================
PEAK MEMORY USAGE: 106.25 KB
---------------- CPU TIME BREAKDOWN ----------------
         8601800 function calls (6001599 primitive calls) in 15.698 seconds

   Ordered by: internal time
   List reduced from 69 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
2600400/200   10.063    0.000   15.639    0.078 2829344861.py:41(_extract)
  2600200    5.278    0.000    5.278    0.000 {method 'extend' of 'list' objects}
  3400600    0.348    0.000    0.348    0.000 {built-in method builtins.isinstance}
      200    0.003    0.000   15.695    0.078 2829344861.py:38(extract_values_from_list)
        3    0.002    0.001    0.052    0.017 selectors.py:451(select)
        1    0.002    0.002   13.151   13.151 2829344861.py:96(run_custom_test)
      200    0.001    0.000    0.001    0.000 {method 'split' of 'str' objects}
        1    0.000    0.000   15.697   15.697 282934486